<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c37-header.png" alt="Nextia Learning · Recommendation and Personalisation: Show Each Person What Matters" width="100%">

# Item-item collaborative filtering

**[Item-item collaborative filtering](https://learning.nextia-ai.com/courses/recommendations/m04/item-item-collaborative-filtering/)** · Recommendation and Personalisation: Show Each Person What Matters · Module 4, lesson 2 of 3 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** score products from everything a customer touched, explain each score, and find the customers it fails.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells (about 8 MB). No account and no key. |
| **You should know** | Cosine similarity of two products, from [People also bought](https://learning.nextia-ai.com/courses/recommendations/m04/people-also-bought/). |
| **Tested** | Python 3.12 with NumPy 2.5.4, pandas 3.1.1, SciPy 1.18.1 and matplotlib 3.11.2 (implicit 0.7.3 or LightGBM 4.7.0 where the lesson needs it), in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/recommendations/m04/item-item-collaborative-filtering/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/M04-L02-item-item-collaborative-filtering/item-item-collaborative-filtering-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: numpy 2.5.4, pandas 3.1.1, scipy 1.18.1, and matplotlib for the pictures. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again.

In [ ]:
%pip install -q --disable-pip-version-check numpy==2.5.4 pandas==3.1.1 scipy==1.18.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
from importlib.metadata import version
for name in ['numpy', 'pandas', 'scipy']:
    print(name, version(name))

## Setup: get the Larkfield data

Run the next cell. It downloads the files that this lesson needs (about 7 MB) into a folder `larkfield-recs` and checks each file's checksum. It needs an internet connection and no account, and takes about 20 seconds.

You should see `Ready:` and the number of files. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-recs` (in Colab: the **Files** panel on the left), then run the cell again.

Where the data comes from: Larkfield is an invented shop, and its shop log is **synthetic**: our code made the products, the customers and every view, basket and order, with a model of customer behaviour. The [dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C37/data/dataset-card.md) explains how.

In [ ]:
# Setup: download the files into larkfield-recs/ and check them.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C37/"
FILES = {
    "data/products.csv": "3da37a3809f6067da12eb5778b8abcf809545599db6959208c72a790861407ff",
    "data/events.csv.gz": "3f46ddfff683586ab1548bc0e6756d85a439a1de941e6de3030edc383d06d1b3",
    "data/splits.json": "c58597803ced6fa787ff23c86d8ee6e9f0c1d7024c0ec109d2cca25caf6213e6",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-recs").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if not (path.exists() and sha256(path) == expected):
        local = HERE / name  # a copy next to the notebook, if there is one
        if local.exists():
            shutil.copy(local, path)
        else:
            download(LABS + name, path)
        if sha256(path) != expected:
            raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-recs and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
print(f"Ready: {len(FILES)} files in larkfield-recs.")

## Setup: helpers

The next cell loads NumPy as `np`, pandas as `pd`, SciPy's sparse matrices as `sp` and matplotlib. It defines the course's small helpers, the same ideas as the course project:

- `products()`, `customers()` and `events()` read the data. `events()` has one row per view, basket add or order, with the column `day` (days since 1 February 2025).
- `split(period)` gives the events before the period (`train`, to learn from) and, for each customer who ordered in the period, the products they ordered (`truth`). The periods are `"validation"`, `"test"` and `"launch"`.
- `recall_at_k(list, ordered, k)`, `ndcg_at_k(list, ordered, k)` and `coverage(lists, n_products)` measure suggestion lists.
- `show(ids)` prints products with their title and subcategory.

It also defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the time split, the measures, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
START = pd.Timestamp("2025-02-01")

def products():
    """One row per product (2,994)."""
    df = pd.read_csv("data/products.csv", keep_default_na=False)
    for col in ("age_restricted", "sensitive", "in_stock"):
        df[col] = df[col].astype(str).eq("True")
    return df

def customers():
    """One row per customer (40,000)."""
    return pd.read_csv("data/customers.csv")

def events():
    """One row per event: view, basket or order. day = days since 1 February 2025."""
    ev = pd.read_csv("data/events.csv.gz", parse_dates=["timestamp"])
    ev["day"] = (ev["timestamp"] - START).dt.days
    return ev

def split(period, ev=None):
    """train = the events before the period; truth = {customer: set of products ordered in the period}."""
    ev = events() if ev is None else ev
    first, last = json.load(open("data/splits.json"))[period]
    a, b = (pd.Timestamp(first) - START).days, (pd.Timestamp(last) - START).days
    train = ev[ev["day"] < a]
    target = ev[(ev["day"] >= a) & (ev["day"] <= b) & (ev["event"] == "order")]
    truth = target.groupby("customer_id")["product_id"].agg(set).to_dict()
    return train, truth

def recall_at_k(rec, ordered, k=10):
    """Share of the ordered products that are in the first k suggestions."""
    return len(set(list(rec)[:k]) & set(ordered)) / len(ordered)

def ndcg_at_k(rec, ordered, k=10):
    """DCG of the list divided by the best possible DCG (discount log2(place + 1))."""
    gains = [1.0 if p in ordered else 0.0 for p in list(rec)[:k]]
    dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
    ideal = sum(1 / np.log2(i + 2) for i in range(min(len(ordered), k)))
    return dcg / ideal

def coverage(lists, n_products, k=10):
    """Share of the catalogue that appears in at least one list."""
    return len({p for rec in lists for p in list(rec)[:k]}) / n_products

def rounded(x, n=3):
    """x rounded to n places (None or ... unchanged: the task is not done yet)."""
    return x if x is None or x is ... else round(float(x), n)

def show(ids, n=10):
    """Print products: place, ID, title, subcategory."""
    global _PR
    if "_PR" not in globals():
        _PR = products().set_index("product_id")
    for i, p in enumerate(list(ids)[:n], 1):
        print(f"{i:2}. {p}  {_PR.loc[p, 'title'][:52]:<52}  {_PR.loc[p, 'subcategory']}")

def fingerprint(value):
    def norm(v):
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 3)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. A score from a customer's history

Item-item collaborative filtering scores every product from the products a customer already touched. Each touched product votes for its neighbours. The vote is the weight of the customer's event times the similarity of the two products.

> **Predict.** Kim ordered a tent and viewed a lantern. Which product gets the highest score: the sleeping bag or the gloves?

In [ ]:
# A tiny example we made: cosine similarity of five products (from lesson 1's basket table)
sim_tiny = pd.DataFrame(
    [[0,     0.707, 0.577, 0.567, 0    ],
     [0.707, 0,     0.408, 0.267, 0    ],
     [0.577, 0.408, 0,     0.436, 0    ],
     [0.567, 0.267, 0.436, 0,     0.535],
     [0,     0,     0,     0.535, 0    ]],
    index=["Tent", "Sleeping bag", "Lantern", "Gloves", "Watering can"],
    columns=["Tent", "Sleeping bag", "Lantern", "Gloves", "Watering can"])
kim = pd.Series({"Tent": 5.0, "Lantern": 1.0})      # Kim ordered a tent (weight 5) and viewed a lantern (weight 1)
score = kim @ sim_tiny.loc[kim.index]               # for each product: sum of (weight x similarity)
print(score.drop(kim.index).sort_values(ascending=False).round(3))

## 2. The similarity of Larkfield's products

The same cosine similarity as in lesson 1, now for all 2,994 products: the customers who touched both products, divided by the square root of the two products' customer counts. Each product keeps only its 50 strongest neighbours. It takes a few seconds.

In [ ]:
ev = events()
train, truth = split("test", ev)
pr = products()
pid = pr["product_id"].to_numpy()
pos = {p: i for i, p in enumerate(pid)}
cid = np.unique(ev["customer_id"])
cpos = {c: i for i, c in enumerate(cid)}
c_idx = train["customer_id"].map(cpos).to_numpy()
p_idx = train["product_id"].map(pos).to_numpy()
B = sp.csr_matrix((np.ones(len(train)), (c_idx, p_idx)), shape=(len(cid), len(pid)))
B.data[:] = 1                                       # touched or not
n = np.asarray(B.sum(axis=0)).ravel()               # customers who touched each product
co = (B.T @ B).toarray()                            # customers who touched both products
np.fill_diagonal(co, 0)
with np.errstate(divide="ignore", invalid="ignore"):
    sim = np.nan_to_num(co / np.sqrt(np.outer(n, n)))
# keep the 50 strongest neighbours of each product, set the rest to 0
weak = np.argpartition(-sim, 50, axis=1)[:, 50:]
np.put_along_axis(sim, weak, 0, axis=1)
print(sim.shape, f"{(sim > 0).sum():,} neighbour links kept")

## 3. A customer's history, with weights

An order counts 5, a basket add 3 and a view 1. Each weight is halved every 60 days before the test period, so old events count less. Here is customer C100123.

In [ ]:
first_day = (pd.Timestamp(json.load(open("data/splits.json"))["test"][0]) - START).days
w = train["event"].map({"view": 1.0, "basket": 3.0, "order": 5.0}).to_numpy()
w = w * 0.5 ** ((first_day - train["day"].to_numpy()) / 60)     # halved every 60 days
H = sp.csr_matrix((w, (c_idx, p_idx)), shape=(len(cid), len(pid)))
H.sum_duplicates()
row = H[cpos["C100123"]]
mine = pd.DataFrame({"product_id": pid[row.indices], "weight": row.data.round(2)}).sort_values("weight", ascending=False)
print(mine.merge(pr[["product_id", "title", "subcategory"]]).head(8).to_string(index=False))

## 4. Your turn: the first suggestion

Compute the scores of every product for C100123: their row of `H` times the similarity matrix `sim`. Then find the product ID with the highest score.

In [ ]:
scores = ...    # the customer's weights (one row of H) times the similarity matrix
top = ...       # the product ID with the highest score
print(top)

In [ ]:
expect_hash('top', top, '354e3e69866bd960')

## 5. Every test customer, by segment

Now score the 2,118 test customers. Only products on sale can be suggested. When a customer has fewer than 10 products with a score, the trending list fills the empty places. It takes about half a minute.

In [ ]:
added = (pd.to_datetime(pr["added"]) - START).dt.days.to_numpy()
on_sale = added <= first_day
orders = train[train["event"] == "order"]
recent = orders[orders["day"] >= first_day - 28].drop_duplicates(["customer_id", "product_id"])
pop28 = recent["product_id"].map(pos).value_counts()
trending = [p for p in pop28.index if on_sale[p]]

users = sorted(truth)
rows = np.array([cpos[u] for u in users])
S = np.asarray((H[rows] @ sim))
S[:, ~on_sale] = -np.inf
S[S <= 0] = -np.inf
lists = []
for s in S:
    best = [int(j) for j in np.argsort(-s, kind="stable")[:10] if np.isfinite(s[j])]
    best += [p for p in trending if p not in best][: 10 - len(best)]   # fill empty places from trending
    lists.append(pid[best])
before = orders.groupby("customer_id")["session_id"].nunique().reindex(users, fill_value=0).to_numpy()
segment = np.where(before == 0, "new", np.where(before <= 2, "occasional", "loyal"))
recall = np.array([recall_at_k(l, truth[u]) for l, u in zip(lists, users)])
table = pd.DataFrame({"segment": segment, "item-cf": recall})
print("item-cf recall@10, all customers:", round(recall.mean(), 3))
print(table.groupby("segment")["item-cf"].mean().reindex(["new", "occasional", "loyal"]).round(3))

## 6. Why occasional customers get poor lists

Occasional customers ordered once or twice before the period. How long ago was that?

In [ ]:
last = orders.groupby("customer_id")["day"].max().reindex(users)
age = pd.DataFrame({"segment": segment, "days since the last order": first_day - last.to_numpy()})
print(age.groupby("segment")["days since the last order"].median().reindex(["occasional", "loyal"]))

> **Check.** Item-cf finds 0.088 of the ordered products on average, but only 0.045 for occasional customers. Their last order is a median of 38 days old (loyal customers: 22 days). Their neighbours are the complements of an old purchase. The project prints the same numbers: `recs evaluate item-cf --by segment`, and `recs explain C100123` shows the first suggestion, P10065.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [User-user and its limits](https://learning.nextia-ai.com/courses/recommendations/m04/user-user-and-its-limits/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/recommendations/m04/item-item-collaborative-filtering/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The Larkfield shop log is synthetic: our code made the products, the customers and everything they did, with a model of customer behaviour. No real customer is in this data.